In [ ]:
import pandas as pd
import missingno as msno
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import zscore
from sklearn.ensemble import IsolationForest
from datetime import datetime
import calendar
from dateutil import parser
import re

In [ ]:
# --- Portable paths (added for the public repository) -----------------------
# Raw workbooks are confidential and are NOT part of this repository.
# Place them locally in data/raw/ (git-ignored) or set AQUAVOLTA_DATA_DIR.
import os
from pathlib import Path

DATA_DIR = Path(os.environ.get("AQUAVOLTA_DATA_DIR", "../../data/raw"))
OUTPUT_DIR = Path(os.environ.get("AQUAVOLTA_OUTPUT_DIR", "../../data/processed"))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
res = pd.read_excel(DATA_DIR / "DATA FORMATION BI 2023.xlsx",sheet_name='EXPORT', header=4)
print(res)

In [ ]:
print(res.columns)

In [ ]:
print(res.shape)

In [ ]:
print(res.info())

In [ ]:
msno.bar(res)

In [ ]:
res["comport"]=(res['Recouvrement €']/res['CA €'])*100
print(res['comport'])

In [ ]:
res['comport'].mean()

In [ ]:
# NOTE: client name anonymized for the public repository (use the real label locally)
clients_cibles = ['CLIENT_A']
res_filtré = res[res['Nom du client'].isin(clients_cibles)].copy()

# Calculer le comportement uniquement pour ces clients
res_filtré['comport'] = (res_filtré['Recouvrement €'] / res_filtré['CA €']) * 100

# Afficher le résultat
print(res_filtré[['Nom du client', 'comport']])

In [ ]:
r1 = res_filtré['comport'].mean(skipna=True)
print(r1)

In [ ]:
res.loc[11,"Recouvrement €"]=res.loc[11,'CA €']*r1
print(res.loc[11,"Recouvrement €"])

In [ ]:
# NOTE: client name anonymized for the public repository (use the real label locally)
clients_cibles = ['CLIENT_B']
res_filtré = res[res['Nom du client'].isin(clients_cibles)].copy()

# Calculer le comportement uniquement pour ces clients
res_filtré['comport'] = (res_filtré['Recouvrement €'] / res_filtré['CA €']) * 100

# Afficher le résultat
print(res_filtré[['Nom du client', 'comport']])

In [ ]:
r2=res_filtré['comport'].mean(skipna=True)
print(r2)

In [ ]:
res.loc[28,"Recouvrement €"]=res.loc[28,'CA €']*r2
print(res.loc[11,"Recouvrement €"])

In [ ]:
print(res.info())

In [ ]:
res = res.drop('comport', axis=1)
print(res.info())

In [ ]:
import pandas as pd
from datetime import datetime
import re

def fix_date(date_str):
    if pd.isna(date_str):
        return None

    d = str(date_str).strip()
    d = d.replace("  ", " ")   # supprimer espaces doubles

    # ✅ Cas spécial : "YYYY-MM-DD 00:00:00"
    if re.match(r"^\d{4}-\d{2}-\d{2} 00:00:00$", d):
        dt = datetime.strptime(d, "%Y-%m-%d %H:%M:%S")
        return dt.strftime("%d/%m/%Y")

    # si contient heure → on garde juste date
    if re.search(r"\d{2}:\d{2}:\d{2}", d):
        d = d.split(" ")[0]

    # séparer jour/mois/année
    parts = re.split(r"[-/]", d)
    if len(parts) != 3:
        return d   # garder comme il est

    day, month, year = parts

    # corriger année mal écrite
    year = year.replace(" ", "")
    if len(year) == 2:  
        year = "20" + year
    elif len(year) != 4:
        return d   # garder comme il est

    try:
        day = int(day)
        month = int(month)
        year = int(year)
    except:
        return d   # garder comme il هو

    # corriger mois hors limite
    if month > 12:
        month = 12
    if month < 1:
        month = 1

    # max jours par mois
    max_days = {
        1: 31, 2: 29 if (year % 4 == 0 and (year % 100 != 0 or year % 400 == 0)) else 28,
        3: 31, 4: 30, 5: 31, 6: 30,
        7: 31, 8: 31, 9: 30, 10: 31,
        11: 30, 12: 31
    }

    if day > max_days[month]:
        day = max_days[month]
    if day < 1:
        day = 1

    fixed_date = datetime(year, month, day)
    return fixed_date.strftime("%d/%m/%Y")

# 🔹 Appliquer sur ton DataFrame
res["DATE"] = res["DATE"].apply(fix_date)
print(res)

In [ ]:
res.to_excel(OUTPUT_DIR / "OUTOUT___EXPORT2023.xlsx", index=False)

In [ ]:
res = pd.read_excel(DATA_DIR / "DATA FORMATION BI 2023.xlsx",sheet_name='LOCAL', header=4)
print(res)

In [ ]:
print(res.columns)

In [ ]:
print(res.info())

In [ ]:
msno.bar(res)

In [ ]:
res=res.rename(columns={"date":"type"})
print(res)

In [ ]:
mode=res[['NOM DU CLIENT']].mode().iloc[0]
for coll in mode.index:
    res[coll]=res[coll].fillna(mode[coll])
print(res.info())

In [ ]:
res['tva 19%']=res['HT']*0.19
print(res['tva 19%'])

In [ ]:
res['TTC']=res['HT']*1.19
print(res['TTC'])

In [ ]:
print(res.describe())

In [ ]:
res["Date"] = pd.to_datetime(res["Date"]).dt.strftime("%d/%m/%Y")
print(res)

In [ ]:
res.to_excel(OUTPUT_DIR / "OUTPUT___LOCAL2023.xlsx", index=False)

In [ ]:
res = pd.read_excel(DATA_DIR / "DATA FORMATION BI 2023.xlsx",sheet_name='DEPENSES', header=6)
print(res)

In [ ]:
print(res.columns)

In [ ]:
print(res.shape)

In [ ]:
print(res.info())

In [ ]:
msno.bar(res)

In [ ]:
res['TTC']=res['HT']+res['TVA']
print(res['TTC'])

In [ ]:
print(res.info())

In [ ]:
res["DATE"] = pd.to_datetime(res["DATE"]).dt.strftime("%d/%m/%Y")
print(res)

In [ ]:
res["DEPARTEMENT"] = res["DEPARTEMENT"].replace({
    "ELEC": "ELECTRICITE",
    "GCIVILE": "GENI CIVIL"
})
print(res)

In [ ]:
res.to_excel(OUTPUT_DIR / "OUTPUT___DEPENSE2023.xlsx", index=False)

In [ ]:
res = pd.read_excel(DATA_DIR / "DATA FORMATION BI 2023.xlsx",sheet_name='SALAIRES', header=6)
print(res)

In [ ]:
print(res.columns)

In [ ]:
print(res.info())

In [ ]:
msno.bar(res)

In [ ]:
count_duplicates = res.duplicated().sum()

print(count_duplicates)

In [ ]:
mois_map = {
    "JANVIER": "01",
    "FEVRIER": "02",
    "MARS": "03",
    "AVRIL": "04",
    "MAI": "05",
    "JUIN": "06",
    "JUILLET": "07",
    "AOUT": "08",
    "SEPTEMBRE": "09",
    "OCTOBRE": "10",
    "NOVEMBRE": "11",
    "DECEMBRE": "12"
}
res["date"] = res["MOIS"].map(mois_map)  
res["date"] = "01/" + res["date"] + "/2023"
print(res[["MOIS", "date"]].head())

In [ ]:
res["DEPARTEMENT"] = res["DEPARTEMENT"].replace({
    "ELEC": "ELECTRICITE",
    "GCIVILE": "GENI CIVIL"
})
print(res)

In [ ]:
res.to_excel(OUTPUT_DIR / "OUTPUT____SALAIRE2023.xlsx", index=False)